# 02 Elevation change rates and uncertainties

Elevation change rates between the five DEMs (1999, 2002, 2012, 2019, 2022), their uncertainties (heteroscedasticity from slope/curvature binning on stable terrain, spatial correlation from a sum of Gaussian and spherical variograms; Hugonnet et al., 2022, xDEM) and the variogram models.

Writes to `../Data_Paper/elevation_change/`, for 2002_1999, 2012_2002, 2019_2012, 2022_2019 and 2022_1999:
- `dhdt_<end>_<start>.tif`, `dhdt_sig_<end>_<start>.tif` (float32, deflate, nodata NaN)
- `<end>_<start>_variogram.csv`

In [ ]:
import os
import geoutils as gu
import xdem
import numpy as np
import pandas as pd

## DEMs, stable terrain and time intervals
All DEMs on the 2019 grid. Stable terrain: off-glacier (RGI 6.0), slope < 20°, 100-1500 m a.s.l. (2012 DEM), |2012-2002| < 5 m.

In [ ]:
dems = {}

dems[2019] = xdem.DEM('../Data_Paper/DEMs/d2019_icesat2.tif')
dems[2019].set_nodata(np.nan)
dems[2022] = xdem.DEM('../Data_Paper/DEMs/d2022_icesat2.tif').reproject(dems[2019])
dems[2022].set_nodata(np.nan)
dems[2012] = xdem.DEM('../Data_Paper/DEMs/d2012_coregistered.tif').reproject(dems[2019])
dems[2012].set_nodata(np.nan)
dems[1999] = xdem.DEM('../Data_Paper/DEMs/d1999_coregistered.tif').reproject(dems[2019])
dems[1999].set_nodata(np.nan)
dems[2002] = xdem.DEM('../Data_Paper/DEMs/d2002_coregistered.tif').reproject(dems[2019])
dems[2002].set_nodata(np.nan)

# For each entry, set values <=0 to NaN
for year in dems:
    dems[year].data[dems[year].data <= 0] = np.nan


list_years = sorted(dems.keys())
glacier_outlines = gu.Vector('../External_Data/RGI60/01_rgi60_Alaska.shp')

glacier_mask = glacier_outlines.create_mask(dems[2012])
stable_terrain = ~glacier_outlines.create_mask(dems[2012])
# Remove terrain where slope > 20 degrees, where dem is below 100m (remove trees) and above 1500m (remove unstable slopes), and where 2012-2002 is too big (glaciers unacounted for)
slope = dems[2012].slope()
slope_mask = slope.data < 20
# Combine the inlier mask and slope mask
stable_terrain.data = stable_terrain.data & slope_mask & (dems[2012].data > 100) & (dems[2012].data < 1500)
# Remove invalid entries from the inlier mask
stable_terrain.data[dems[2012].data.mask==True]=False
stable_terrain.data[dems[2002].data.mask==True]=False
stable_terrain.data = np.logical_and(stable_terrain.data,np.abs((dems[2012].data.data - dems[2002].data.data) < 5))


time_intervals = []
# Calculate the time intervals (SRTM reference date 1 September 1999, as in the paper)
time_intervals.append((pd.to_datetime('2002-08-04') - pd.to_datetime('1999-09-01')).days / 365.25)
time_intervals.append((pd.to_datetime('2012-08-30') - pd.to_datetime('2002-08-04')).days / 365.25)
time_intervals.append((pd.to_datetime('2019-08-01') - pd.to_datetime('2012-08-30')).days / 365.25)
time_intervals.append((pd.to_datetime('2022-08-01') - pd.to_datetime('2019-08-01')).days / 365.25)
time_intervals.append((pd.to_datetime('2022-08-01') - pd.to_datetime('1999-09-01')).days / 365.25)

years_unc = sorted(dems.keys())

## Bin limits: 95th percentile of on-glacier slope and curvature over all DEMs

In [ ]:
max_slope = []
max_curv = []
for yr in dems:
    slope = dems[yr].slope().data[glacier_mask.data]
    curv = dems[yr].curvature().data[glacier_mask.data]
    max_slope.append(np.nanpercentile(slope.data, 95))
    max_curv.append(np.nanpercentile(curv.data, 95))

max_slope = np.array(max_slope).max()
max_curv = np.array(max_curv).max()

## Error model: heteroscedasticity (slope, curvature) and variogram of standardized dh on stable terrain

In [ ]:
def error_dh(dems, yr1, yr2, stable_terrain, max_slope, max_curv):

    attributes = ["slope", "curvature"]

    # Get elevation differences and stable terrain mask
    dh = dems[yr2] - dems[yr1]

    # Derive slope and curvature
    slope, curv = dems[yr2].get_terrain_attribute(attribute=attributes)

    # Use only array of stable terrain
    dh_arr = dh[stable_terrain]
    slope_arr = slope[stable_terrain]
    curv_arr = curv[stable_terrain]

    # Estimate the variable error by bin of slope and curvature
    df_h = xdem.spatialstats.nd_binning(
        dh_arr, list_var=[slope_arr, curv_arr], list_var_names=attributes,
        list_var_bins=[np.linspace(0, max_slope, 2*(int(max_slope)+1)), np.linspace(-max_curv, max_curv, 10*(int(2*max_curv)+1))]
    )

    # Plot 2D binning
    xdem.spatialstats.plot_2d_binning(df_h, "slope", "curvature", "nmad", "Slope (degrees)", "Curvature (100 m-1)", "NMAD (m)")


    # Derive a numerical function of the measurement error
    sig_dh_func = xdem.spatialstats.interp_nd_binning(df_h, list_var_names=["slope", "curvature"])
    sig_dh_arr = sig_dh_func((slope.data, curv.data))

    # Convert to raster
    sig_dh = dh.copy(new_array=sig_dh_arr)

    # Standardize the data
    z_dh = dh / sig_dh
    # Mask values to keep only stable terrain
    z_dh.set_mask(~stable_terrain)

    # Sample empirical variogram
    df_vgm = xdem.spatialstats.sample_empirical_variogram(values=z_dh, subsample=500, n_variograms=5, random_state=42)


    # Fit the sum of a gaussian and spherical model
    func_sum_vgm, params_variogram_model = xdem.spatialstats.fit_sum_model_variogram(
        list_models=["Gaussian", "Spherical"], empirical_variogram=df_vgm
    )
    # Plot empirical and modelled variogram
    xdem.spatialstats.plot_variogram(df_vgm, [func_sum_vgm], ["Sum of gaussian and spherical"], xscale="log")


    return dh, sig_dh, params_variogram_model

## dh/dt and σ(dh/dt) for consecutive pairs and for 2022-1999

In [ ]:
dhdt = {}
dhdt_sig= {}
dhdt_variograms = {}

for i in range(1,len(years_unc)):

    # Calculate the error for the current year compared to the previous year
    dh, sig_dh, params_variogram_model = error_dh(dems, years_unc[i-1], years_unc[i], stable_terrain, max_slope, max_curv)

    # Store the variogram model parameters
    dhdt_variograms[f"{years_unc[i]}-{years_unc[i-1]}"] = params_variogram_model

    # Store the elevation difference and uncertainty
    dhdt[f"{years_unc[i]}-{years_unc[i-1]}"] = dh/time_intervals[i-1]
    dhdt_sig[f"{years_unc[i]}-{years_unc[i-1]}"] = sig_dh/time_intervals[i-1]

# Now for 2022-1999

dh, sig_dh, params_variogram_model = error_dh(dems, 1999, 2022, stable_terrain, max_slope, max_curv)

# Store the variogram model parameters
dhdt_variograms[f"{2022}-{1999}"] = params_variogram_model

# Store the elevation difference and uncertainty
dhdt[f"{2022}-{1999}"] = dh/time_intervals[-1]
dhdt_sig[f"{2022}-{1999}"] = sig_dh/time_intervals[-1]

## Write rasters and variograms

In [ ]:
out_dir = '../Data_Paper/elevation_change'
os.makedirs(out_dir, exist_ok=True)

for period in dhdt:
    end, start = period.split('-')
    dhdt_variograms[period].to_csv(f"{out_dir}/{end}_{start}_variogram.csv")
    dhdt[period].astype(np.float32).save(f"{out_dir}/dhdt_{end}_{start}.tif", nodata=np.nan, compress='deflate')
    dhdt_sig[period].astype(np.float32).save(f"{out_dir}/dhdt_sig_{end}_{start}.tif", nodata=np.nan, compress='deflate')